# Test a public agent before you spend a slot on it

Only your two most recent submissions are alive, so every submission you spend on an
untested public agent costs you a live slot for the day it takes to converge. This
notebook is the cheap test I run first, and one measurement that shows why it is worth
running.

The short version: **a published notebook is not the same object as the agent its author
is fielding**, and the gap can be total rather than marginal.

## The measurement

I took a public agent whose author sat at **rank 90** on the day, extracted its `main.py`
without executing the notebook, and played it against a fixed reference agent for
**192 games in the official environment**, swapping seats on odd seeds so no side keeps
the first-move advantage.

```
published agent   0
reference       192
draws             0      crashes 0
mean end money    74,419  vs  91,239
```

Zero wins in 192 games, z = -13.9. This is not a close call that a bigger sample would
overturn; the agent simply does not play the way its author's rating suggests.

Two things it is not evidence of. It is not evidence that the author's fielded agent is
weak: the author's rating is real and was earned by something. And it is not a claim that
the notebook is wrong about what it does; the code ran clean, produced legal actions for
all 720 turns, and finished every game.

## Why the two can differ

destbreso [showed this directly](https://www.kaggle.com/code/destbreso/97-predictable-extracting-decision-trees):
replaying one author's published code turn by turn against that author's own recorded
ladder games, the two disagree at turn 0 and differ on 697 of 719 turns. A sanitised
public release is entirely legitimate and completely invisible from the outside - the
rating, the forum, and the notebook's own documentation all look the same either way.

So the author's rating is a prior about the author, not a measurement of the artefact you
just downloaded. The only thing that measures the artefact is playing it.

## The test

Sample size first, because this is where it is easy to fool yourself. Against an opponent
you truly beat 55% of the time, **664 paired games** is where the expected result lands
exactly on the p < 0.01 threshold - that is a coin flip on whether any single run of that
length clears it. Wanting it to clear 80% of the time costs about 1,170. At 192 games the
smallest edge you can resolve is around **60%**, so triage is what 192 games buys you:
*is this broken or roughly competitive*, not *is it 1% better*. Do not read a 105-87
result as a finding.

In [ ]:
import math

def games_needed(true_rate, alpha=0.01, power=0.5):
    """Paired games needed to separate `true_rate` from 0.5.

    power=0.5 is the point where the expected result lands exactly on the threshold;
    power=0.8 is where a single run clears it four times out of five.
    """
    z_a = 2.576 if alpha == 0.01 else 1.96
    z_b = {0.5: 0.0, 0.8: 0.842, 0.9: 1.282}[power]
    return math.ceil(((z_a + z_b) / (2 * (true_rate - 0.5))) ** 2)

print(f"{'true win rate':>14}  {'p<0.01, 50% power':>18}  {'p<0.01, 80% power':>18}")
for r in (0.55, 0.60, 0.65, 0.70, 0.80):
    print(f"{r:>14.2f}  {games_needed(r):>18,}  {games_needed(r, power=0.8):>18,}")

def z_score(wins, draws, n):
    """Draws count as half. Two-sided z against a fair coin."""
    points = wins + draws / 2
    return (points - n / 2) / math.sqrt(n / 4)

print()
print('192 games, 0 wins   :  z =', round(z_score(0, 0, 192), 1))
print('192 games, 118 wins :  z =', round(z_score(118, 0, 192), 1))
print('192 games, 105 wins :  z =', round(z_score(105, 0, 192), 1), ' <- not a finding')

## Three details that decide whether the number means anything

**Swap seats.** Run every seed twice with the sides exchanged, or alternate by seed
parity. Seat asymmetry in this game is large enough to manufacture a result on its own.

**Count draws as half.** Knob variants routinely produce hundreds of identical games. A
run that comes back 33-23 with 328 draws has not told you the knob is good, it has told
you the knob was never reached. Excluding draws and reporting 59% hides exactly the thing
you needed to see.

**Watch for zero draws.** The mirror of the above. When a change produces a different
outcome in every single game, it is genuinely on the execution path, whatever the win rate
turns out to be.

In [ ]:
# The comparison in the form worth reporting: half-point rate, sample size, z, and
# whether the change is even on the execution path.
def report(label, wins, losses, draws):
    n = wins + losses + draws
    half = (wins + draws / 2) / n
    z = z_score(wins, draws, n)
    reach = 'yes' if draws < n * 0.1 else 'NO - mostly identical games'
    print(f"{label:<24} {wins:>4}-{losses:<4}-{draws:<4}  half {half:6.1%}  z {z:6.2f}  on path: {reach}")

report('published agent', 0, 192, 0)
report('a knob variant', 33, 23, 328)
report('a real change', 118, 74, 0)

## What I do with the result

A public agent that loses everything against a reference is not worth a submission slot,
however high its author sits. A public agent that wins convincingly is worth one slot -
and then the ladder, not the local panel, is what decides, because the local panel is one
opponent and the ladder is the whole field.

The companion notebook to this one,
[a ruler for agent changes](https://www.kaggle.com/code/rogerrogerroger3r/kaggriculture-a-ruler-for-agent-changes),
has the paired harness and the free way to build the C++ chassis without a compiler on
your machine.

Credit to **destbreso** for the published-versus-fielded case, which is the reason this
test exists rather than being paranoia.